# AgriSmart: train the crop-ID and disease models on Colab

This notebook downloads public leaf datasets, trains one model per supported crop (**Apple, Tomato, Capsicum**) plus the crop identifier,
and saves the `.onnx` files to your Google Drive. Then you copy them into `services/api/models/`.

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. On your PC run `python scripts/make_colab_bundle.py` (repo root) to create `agrismart_colab_bundle.zip`.
3. Run the cells below **in order**. Expect roughly 3-5 GB of downloads and 1-2 hours in total.
   Keep the tab open; if Colab disconnects, re-run from the top (downloads and finished models are cached on Drive/disk).

**Honest expectations:** accuracy printed here is measured on held-out images from the *same* datasets.
PlantVillage-style photos are far cleaner than farm photos, so real-world accuracy will be lower.
Read `docs/models.md`.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MODELS_OUT = "/content/drive/MyDrive/agrismart_models"   # trained .onnx files land here
!mkdir -p "$MODELS_OUT"
print("Models will be saved to:", MODELS_OUT)

## 2. Upload the bundle (`agrismart_colab_bundle.zip`)

In [ ]:
from google.colab import files
import zipfile, io
uploaded = files.upload()
name = next(iter(uploaded))
zipfile.ZipFile(io.BytesIO(uploaded[name])).extractall("/content/agrismart")
%cd /content/agrismart
!ls scripts services/api/app/core

In [ ]:
!pip -q install onnx onnxruntime requests

## 3. Download and arrange the datasets (about 10-20 minutes)

Pulls PlantVillage + PlantDoc (GitHub) for **apple, tomato and capsicum**, plus generic non-plant photos and other
plant species (potato, corn, ...) for the crop identifier's `unknown` class.

Wheat, sugarcane and rice/paddy are **not** downloaded: their datasets are not accessible from Colab at the moment,
so those crops are out of the active scope (see `docs/models.md`).

To use a mirror or a local clone, set `AGRISMART_PLANTVILLAGE_SRC` / `AGRISMART_PLANTDOC_SRC` before running the next cell.
If a source cannot be reached the script stops with a clear message instead of continuing with partial data.

In [ ]:
!python scripts/prepare_datasets.py --out-dir /content/data

### 4. Check the dataset report above before training

Every crop should say `OK`. If a crop is `INCOMPLETE`, find more images for the listed classes or remove those
classes from `services/api/app/core/model_catalog.py` (and `app/knowledge/`) and re-bundle.
Training still works for the other crops.

In [ ]:
import os
for crop in sorted(os.listdir("/content/data/disease")):
    for cls in sorted(os.listdir(f"/content/data/disease/{crop}")):
        n = len(os.listdir(f"/content/data/disease/{crop}/{cls}"))
        print(f"{crop:10s} {cls:26s} {n:6d}")

## 5. Train every model

Set `QUICK = True` first for a 5-minute smoke test (1 epoch each), then set it back to `False` for the real run.
Crops that fail (for example an incomplete dataset) are reported at the end instead of stopping everything.

In [ ]:
import subprocess, sys

QUICK = False
EPOCHS, HEAD_EPOCHS = (1, 1) if QUICK else (12, 2)
CROPS = ["apple", "tomato", "capsicum"]   # keep in sync with SUPPORTED_CROPS
failed = []

def train(args, label):
    print(f"\n===== {label} =====")
    cmd = [sys.executable, "scripts/train_disease_model.py", "--out-dir", MODELS_OUT,
           "--epochs", str(EPOCHS), "--head-epochs", str(HEAD_EPOCHS), *args]
    if subprocess.run(cmd).returncode != 0:
        failed.append(label)

for crop in CROPS:
    train(["--mode", "disease", "--crop", crop, "--data-dir", f"/content/data/disease/{crop}"], crop)
train(["--mode", "crop-id", "--data-dir", "/content/data/crop_id"], "crop-id")

print("\nFAILED:", failed or "none")

## 6. Results

In [ ]:
import glob, json, os
print(f"{'model':28s} {'accuracy':>9s} {'macro F1':>9s}  weakest class")
for path in sorted(glob.glob(f"{MODELS_OUT}/*.metrics.json")):
    m = json.load(open(path))
    worst = min(m["per_class"].items(), key=lambda kv: kv[1]["f1"])
    print(f"{os.path.basename(path)[:-13]:28s} {m['accuracy']:9.3f} {m['macro_f1']:9.3f}  {worst[0]} (F1 {worst[1]['f1']:.2f})")
print("\nFiles in Drive:", sorted(os.listdir(MODELS_OUT)))

## 7. Install the models in the app

1. From Google Drive, download `agrismart_models/` and copy every `*.onnx` and `*.meta.json` file into `services/api/models/`.
2. Restart the API and open `http://localhost:8000/api/v1/diagnose/models` - trained crops should say `available`.
3. A crop that stays `weights_missing` (for example one that failed above) still returns `model_unavailable`: nothing is faked.
4. The crop identifier (`crop_id.onnx`) is required for any diagnosis to work.

**Test with real farm photos** (not dataset images). If accuracy is poor, add your own field photos to the datasets and retrain.